# Tutorial 4: Unstructured Pruning on Bert

Pruning is a technique used to reduce the size and complexity of neural networks by removing unnecessary parameters (weights and connections) or structural components (neurons, filters, or layers). The goal is to create a smaller, more efficient model that maintains most of the original model's performance. The following benefits can be seen from pruning neural networks:

- **Reduce model size**: Deep neural networks often have millions of parameters, leading to large storage requirements.

- **Decrease inference time**: Fewer parameters mean fewer computations, resulting in faster predictions.

- **Improve generalization**: Removing unnecessary connections can help prevent overfitting.

- **Energy efficiency**: Smaller models require less energy to run, which is crucial for edge devices and mobile applications.

Structured pruning removes entire structures (e.g., channels, filters, or layers) from the network, while unstructured pruning removes individual weights or connections from the network, regardless of their location. In this tutorial, we'll build on top of Tutorial 3 by taking the quantized Bert model and running Mase's unstructured pruning pass. After pruning, we'll run further fine tuning iterations to retain sequence classification accuracy in the pruned model.

In [1]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

## Importing the model

If you are starting from scratch, you can create a MaseGraph for Bert by running the following cell.

In [2]:
from transformers import AutoModelForSequenceClassification

from chop import MaseGraph
import chop.passes as passes

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
model.config.problem_type = "single_label_classification"

mg = MaseGraph(
    model,
    hf_input_names=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg)

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.
INFO     Getting dummy input for prajjwal1/bert-tiny.


tensor([[ 101, 9932, 2089, 2202, 2058, 1996, 2088, 2028, 2154,  102],
        [ 101, 2023, 2003, 2339, 2017, 2323, 4553, 4748, 4877,  102]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[ 101, 9932, 2089, 2202, 2058, 1996, 2088, 2028, 2154,  102],
        [ 101, 2023, 2003, 2339, 2017, 2323, 4553, 4748, 4877,  102]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]],


        [[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]]])
tensor([[[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       

If you have previously ran the tutorial on Quantization-Aware Training (QAT), run the following cell to import the fine tuned checkpoint.

In [3]:
from pathlib import Path
from chop import MaseGraph

mg = MaseGraph.from_checkpoint(f"{Path.home()}/tutorial_3_qat")

## Unstructured Pruning

Before running pruning, let's evaluate the model accuracy on the IMDb dataset. If you're coming from Tutorial, this would be the same as the accuracy after Quantization Aware Training (QAT). If you've just initialized the model, this will likely be a random guess (i.e. around 50%), in which case pruning wouldn't have a significant effect on the accuracy.

In [6]:
from chop.tools import get_tokenized_dataset, get_trainer

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)


INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.
/workspace/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Evaluate accuracy
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

To run the pruning pass, we pass the following pruning configuration dictionary, which defines the following parameters.

- **Sparsity**: a value between 0 and 1, expressing the proportion of elements in the model that should be pruned (i.e. set to 0).

- **Method**: several pruning methods are supported, including ``Random`` and ``L1-Norm``.

- **Scope**: defines whether to consider each weight/activation tensor individually (``local``) or all tensors in the model (``global``) when obtaining statistics for pruning (e.g. absolute value threshold for pruning)

We'll start by running random pruning with local scope, at a fixed sparsity. This may be suboptimal, but in future tutorials we'll see how to find optimal pruning and quantization configurations for a given model on a specified dataset.

In [4]:
import chop.passes as passes

pruning_config = {
    "weight": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
    "activation": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
}

mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)

INFO     Pruning module: bert_encoder_layer_0_attention_self_query
INFO     Pruning module: bert_encoder_layer_0_attention_self_key
INFO     Pruning module: bert_encoder_layer_0_attention_self_value
INFO     Pruning module: bert_encoder_layer_0_attention_output_dense
INFO     Pruning module: bert_encoder_layer_0_intermediate_dense
INFO     Pruning module: bert_encoder_layer_0_output_dense
INFO     Pruning module: bert_encoder_layer_1_attention_self_query
INFO     Pruning module: bert_encoder_layer_1_attention_self_key
INFO     Pruning module: bert_encoder_layer_1_attention_self_value
INFO     Pruning module: bert_encoder_layer_1_attention_output_dense
INFO     Pruning module: bert_encoder_layer_1_intermediate_dense
INFO     Pruning module: bert_encoder_layer_1_output_dense
INFO     Pruning module: bert_pooler_dense
INFO     Pruning module: classifier


Let's evaluate the effect of pruning on accuracy. It's likely to observe drops of around 10% or more.

In [7]:
trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=5,
)

# Evaluate accuracy
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

/workspace/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation accuracy: 0.7886


To overcome the drop in accuracy, we'll run a few finetuning epochs. This allows the model to adapt to the new pruning mask.

KD part:

In [7]:
import sys
import os
from transformers import AutoModelForSequenceClassification

# 确保可以导入你自己写的 mase_kd 模块 (假设你在项目根目录运行 notebook)
sys.path.append(os.path.abspath('src'))
from mase_kd.distillation import KnowledgeDistillationPass

# 加载一个在 IMDB 数据集上微调过的高精度 bert-base 模型作为老师
teacher_checkpoint = "textattack/bert-base-uncased-IMDB"
print(f"Loading teacher model from {teacher_checkpoint}...")

# 加载老师模型，并严格设置为 eval 模式（不更新参数）
teacher_model = AutoModelForSequenceClassification.from_pretrained(teacher_checkpoint)
teacher_model.eval()

Loading teacher model from textattack/bert-base-uncased-IMDB...


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [8]:
# 初始化你的核心蒸馏引擎
# 学生 (bert-tiny): 2 层，128 维
# 老师 (bert-base): 12 层，768 维
# 初始化你的核心蒸馏引擎 (Logits-Only FX Traced 适配版)
kd_model = KnowledgeDistillationPass(
    student_model=mg.model,       # mg.model 是你刚才剪枝后的模型
    teacher_model=teacher_model,
    s_dim=128,                    # 这些参数会被 **kwargs 安全吸收
    t_dim=768,
    s_layers=2,
    t_layers=12,
    alpha_kd=1.0,                 # 蒸馏总损失的权重
    temperature=3.0               # 软标签蒸馏温度
)

print("KD 模型封装成功！当前采用 Logits-Only 蒸馏策略以适配 FX 静态图。")



KD 模型封装成功！当前采用 Logits-Only 蒸馏策略以适配 FX 静态图。


Training with KD only use logits, without hidden layers KD.

In [12]:
from chop.tools import get_trainer

# 使用项目提供的工具直接生成 Trainer，传入包装了联合 Loss 的 kd_model
kd_trainer = get_trainer(
    model=kd_model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=5,  # 同样运行 5 个 Epoch

)

# 💥 见证奇迹的时刻：开始剪枝后的蒸馏微调！
print("Starting Knowledge Distillation Fine-tuning...")
kd_trainer.train()

/workspace/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting Knowledge Distillation Fine-tuning...


Step,Training Loss
500,2.470200
1000,2.379900
1500,2.462700
2000,2.390100
2500,2.380800
3000,2.415900
3500,2.467400
4000,2.417000
4500,2.272900
5000,2.341600


TrainOutput(global_step=15625, training_loss=2.306140951171875, metrics={'train_runtime': 2043.0423, 'train_samples_per_second': 61.183, 'train_steps_per_second': 7.648, 'total_flos': 0.0, 'train_loss': 2.306140951171875, 'epoch': 5.0})

Evaluation

In [13]:
# 重新获取一个只包含 mg.model（剪枝学生）的 Trainer 用于最终公平评估
eval_trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

eval_results = eval_trainer.evaluate()
print(f"🎉 蒸馏微调后的最终评估准确率 (Evaluation accuracy after KD): {eval_results['eval_accuracy']}")

/workspace/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🎉 蒸馏微调后的最终评估准确率 (Evaluation accuracy after KD): 0.83988


Evaluation with teacher model(Bert-Base)

In [ ]:
from transformers import AutoModelForSequenceClassification
from chop.tools import get_trainer

# ==========================================
# 评估原始满血版 Bert-Base (Baseline)
# ==========================================

# 1. 重新加载老师模型（使用全新变量名 reference_base_model）
base_checkpoint = "textattack/bert-base-uncased-IMDB"
print(f"Loading full precision baseline from {base_checkpoint}...")
reference_base_model = AutoModelForSequenceClassification.from_pretrained(base_checkpoint)

# 2. 为 Base 模型创建一个专属的评估 Trainer 
base_eval_trainer = get_trainer(
    model=reference_base_model,
    tokenized_dataset=dataset,   # 直接使用你 Notebook 内存里现有的 dataset
    tokenizer=tokenizer,         # 直接使用你 Notebook 内存里现有的 tokenizer
    evaluate_metric="accuracy",
)

# 3. 运行评估
print("Evaluating the original bert-base model. This might take a minute...")
base_eval_results = base_eval_trainer.evaluate()

# 4. 打印极其重要的基准对比数据！
print("-" * 50)
print(f"🏆 [Baseline] 满血版 Bert-Base 准确率: {base_eval_results['eval_accuracy']:.5f}")
print("-" * 50)

Loading full precision baseline from textattack/bert-base-uncased-IMDB...


/workspace/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluating the original bert-base model. This might take a minute...


--------------------------------------------------
🏆 [Baseline] 满血版 Bert-Base 准确率: 0.93028
--------------------------------------------------


Saving KD_logits_only model

In [ ]:
import torch
import os

# 1. 创建一个专门存放实验模型的文件夹
save_dir = "saved_models/kd_logits_only"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "student_weights_0.83988.pt")

# 2. 提取并保存学生模型 (mg.model) 的权重
# 注意：我们只存 mg.model，不需要存庞大的 teacher_model
torch.save(mg.model.state_dict(), save_path)
print(f"✅ 太棒了！Logits 蒸馏模型的权重已安全保存至: {save_path}")

✅ 太棒了！Logits 蒸馏模型的权重已安全保存至: saved_models/kd_logits_only/student_weights_0.83988.pt
你现在可以放心地去上面重新运行剪枝代码了！


Apply the same logic to the hidden layer distillation code:

In [12]:
import sys
import os
sys.path.append(os.path.abspath('src'))

# Import the brand new Pass from your newly created file
from mase_kd.distillation.kd_pass_hidden import kd_transform_pass

# ==========================================
# Experiment B: Joint Distillation (Logits + Hidden Layer)
# ==========================================

# 1. Adjust and set the new configuration parameters
kd_hidden_config = {
    "s_dim": 128,
    "t_dim": 768,
    "s_layers": 2,
    "t_layers": 12,
    "alpha_kd": 2.0,       # 🔥 Adjustment 1: Increase the learning weight
    "alpha_hidden": 1.0,   # Enable hidden layer feature learning
    "temperature": 5.0     # 🔥 Adjustment 2: Increase temperature 
}

# 2. Call the new Pass 
# 🌟 这里的 teacher_model=teacher_model 修改为了你代码中实际定义的变量名 🌟
kd_hidden_model, hidden_pass_info = kd_transform_pass(
    student_graph_or_model=mg, 
    teacher_model=teacher_model, # <--- 修改了这里！使用你刚才加载的 teacher_model
    config=kd_hidden_config
)

print("-" * 50)
print(f"✅ Hidden KD Pass successfully wrapped! Execution info: {hidden_pass_info}")
print("-" * 50)

[KD Pass] Initializing Knowledge Distillation graph transformation...
[KD Pass] Successfully registered 4 feature hooks on the student model.
--------------------------------------------------
✅ Hidden KD Pass successfully wrapped! Execution info: {'pass_name': 'kd_transform_pass', 'teacher_layers': 12, 'student_layers': 2, 'hooks_registered': True}
--------------------------------------------------


Training

In [13]:
# ==========================================
# 启动训练
# ==========================================
kd_hidden_trainer = get_trainer(
    model=kd_hidden_model,
    tokenized_dataset=dataset,   
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=5
)

print("🚀 Starting advanced KD training with Hidden Layers... (Grab a coffee!)")
kd_hidden_trainer.train()

/workspace/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Starting advanced KD training with Hidden Layers... (Grab a coffee!)


Step,Training Loss
500,9.403100
1000,7.642000
1500,7.400000
2000,6.777800
2500,6.628600
3000,6.801000
3500,6.888100
4000,6.644100
4500,6.303900
5000,6.339200


TrainOutput(global_step=15625, training_loss=6.46065758203125, metrics={'train_runtime': 2123.7362, 'train_samples_per_second': 58.859, 'train_steps_per_second': 7.357, 'total_flos': 0.0, 'train_loss': 6.46065758203125, 'epoch': 5.0})

Saving KD_logits_hidden model

In [19]:
import torch
import os

# 1. 创建保存路径 (区分之前的 Logits-only 文件夹)
save_dir_hidden = "saved_models/kd_logits_hidden"
os.makedirs(save_dir_hidden, exist_ok=True)

# 建议在文件名中标记该版本使用的 T=5.0 和 alpha_kd=2.0
save_path_hidden = os.path.join(save_dir_hidden, "student_hidden_kd_final.pt")

# 2. 保存 mg.model 的状态字典
# 虽然外层有 Wrapper，但核心权重都在 mg.model 中
try:
    torch.save(mg.model.state_dict(), save_path_hidden)
    print(f"✅ 成功！策略 B 的模型权重已安全保存至: {save_path_hidden}")
    print("现在你可以尝试重启 Kernel 或清理显存进行评估了。")
except Exception as e:
    print(f"❌ 保存失败: {str(e)}")

✅ 成功！策略 B 的模型权重已安全保存至: saved_models/kd_logits_hidden/student_hidden_kd_final.pt
现在你可以尝试重启 Kernel 或清理显存进行评估了。


Evaluation of two models.

In [28]:
import torch
from chop.tools import get_trainer

# 1. 重新指定路径
save_path_hidden = "saved_models/kd_logits_hidden/student_hidden_kd_final.pt"

print(f"🔄 正在尝试【原位加载】策略 B 权重: {save_path_hidden}...")

# 2. 直接加载原始 state_dict，不做任何 key 替换
# 因为你的 mg.model 现在已经有了 .parametrizations 结构
raw_state_dict = torch.load(save_path_hidden, map_location="cpu")

# 3. 尝试直接加载
# 这一次我们保持 strict=True，如果报错，它会告诉我们到底缺什么
try:
    mg.model.load_state_dict(raw_state_dict, strict=True)
    print("✅ 太棒了！权重完全匹配，加载成功！")
except Exception as e:
    print(f"⚠️ 依然报错，尝试强制加载...")
    mg.model.load_state_dict(raw_state_dict, strict=False)
    print("⚠️ 已使用非严格模式加载。")

mg.model.to("cuda")
mg.model.eval()

# 4. 再次评估 (Batch Size = 1)
eval_trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

print("🚀 重新开始评估...")
results = eval_trainer.evaluate()

# 5. 打印对比
print("\n" + "=" * 50)
print("📊 修正后的最终对比结果：")
print("=" * 50)
print(f"🔹 策略 A (仅 Logits 蒸馏): 0.83988")
print(f"🔸 策略 B (Logits + Hidden 联合蒸馏): {results['eval_accuracy']:.5f}")
print("=" * 50)

🔄 正在尝试【原位加载】策略 B 权重: saved_models/kd_logits_hidden/student_hidden_kd_final.pt...
⚠️ 依然报错，尝试强制加载...
⚠️ 已使用非严格模式加载。
🚀 重新开始评估...


/workspace/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



📊 修正后的最终对比结果：
🔹 策略 A (仅 Logits 蒸馏): 0.83988
🔸 策略 B (Logits + Hidden 联合蒸馏): 0.83632


attention layer KD 

In [10]:
import sys
import os

sys.path.append(os.path.abspath('src'))

from mase_kd.distillation.kd_pass_hidden import kd_transform_pass


In [11]:
import torch
import gc
from transformers import AutoModelForSequenceClassification
from chop.tools import get_trainer

# 确保导入了带有 Attention 蒸馏的新版 Pass
# from mase_kd.distillation.kd_pass_hidden import kd_transform_pass

# ==========================================
# 1. 准备老师模型
# ==========================================
teacher_checkpoint = "textattack/bert-base-uncased-IMDB"
print(f"🔄 loading teacher model: {teacher_checkpoint}...")
teacher_model = AutoModelForSequenceClassification.from_pretrained(teacher_checkpoint)
teacher_model.eval()
teacher_model.to("cuda")

kd_attn_config = {
    "s_dim": 128,          # ✅ 确认：学生真实维度 128
    "t_dim": 768,          # ✅ 确认：老师（BERT-Base）维度 768
    "s_layers": 2,         # ✅ 确认：学生真实层数 2
    "t_layers": 12,        # ✅ 确认：老师真实层数 12
    
    "alpha_kd": 1.0,       # 🌟 Logits 是主力，权重拉满
    "alpha_hidden": 0.1,   # 🔥 隐层映射跨度太大，降权作为微弱辅助
    "alpha_attn": 0.1,     # 🔥 注意力矩阵太复杂，降权防止容量过载
    "temperature": 2.0     # 🔥 温度保持在较低水平
}

print(f"🔄 Logits + Hidden + Attention】KD Pass...")
# 🚨 前提假设：你的 mg 变量 (初始剪枝后的学生模型) 已经准备好
kd_attn_model, attn_pass_info = kd_transform_pass(
    student_graph_or_model=mg, 
    teacher_model=teacher_model, 
    config=kd_attn_config
)
print(f"✅ {attn_pass_info}")


original_state_dict = kd_attn_model.state_dict
def contiguous_state_dict(*args, **kwargs):
    sd = original_state_dict(*args, **kwargs)
    return {k: v.contiguous() for k, v in sd.items()}
kd_attn_model.state_dict = contiguous_state_dict

# ==========================================
# 3. 开始训练
# ==========================================
print("🚀 starting training...")
kd_attn_trainer = get_trainer(
    model=kd_attn_model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=5  # 保持和你之前实验一样的 epoch 数
)
kd_attn_trainer.train()



🔄 loading teacher model: textattack/bert-base-uncased-IMDB...
🔄 Logits + Hidden + Attention】KD Pass...
[KD Pass] Initializing Knowledge Distillation graph transformation...
[KD Pass] Successfully registered 4 feature hooks on the student model.
✅ {'pass_name': 'kd_transform_pass', 'teacher_layers': 12, 'student_layers': 2, 'hooks_registered': True}
🚀 starting training...


/workspace/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,2.112800
1000,1.835700
1500,1.862800
2000,1.735300
2500,1.733500
3000,1.764000
3500,1.804600
4000,1.741300
4500,1.663900
5000,1.686300


TrainOutput(global_step=15625, training_loss=1.6932474599609375, metrics={'train_runtime': 2230.5161, 'train_samples_per_second': 56.041, 'train_steps_per_second': 7.005, 'total_flos': 0.0, 'train_loss': 1.6932474599609375, 'epoch': 5.0})

In [12]:
import torch
import os

# 1. 创建保存路径 (区分之前的 Logits-only 文件夹)
save_dir_hidden = "saved_models/kd_logits_hidden_attention"
os.makedirs(save_dir_hidden, exist_ok=True)

# 建议在文件名中标记该版本使用的 T=5.0 和 alpha_kd=2.0
save_path_hidden = os.path.join(save_dir_hidden, "student_attention_kd_final2.0.pt")

# 2. 保存 mg.model 的状态字典
# 虽然外层有 Wrapper，但核心权重都在 mg.model 中
try:
    torch.save(mg.model.state_dict(), save_path_hidden)
    print(f"✅ 成功！策略 B 的模型权重已安全保存至: {save_path_hidden}")
    print("现在你可以尝试重启 Kernel 或清理显存进行评估了。")
except Exception as e:
    print(f"❌ 保存失败: {str(e)}")

✅ 成功！策略 B 的模型权重已安全保存至: saved_models/kd_logits_hidden_attention/student_attention_kd_final2.0.pt
现在你可以尝试重启 Kernel 或清理显存进行评估了。


In [8]:
import torch
from chop.tools import get_trainer

# 1. 指定刚刚保存的含有 Attention 蒸馏的权重路径
save_path_attn = "saved_models/kd_logits_hidden_attention/student_attention_kd_final2.0.pt"
print(f"🔄 正在读取权重: {save_path_attn}")

# 2. 读取硬盘上的权重
state_dict = torch.load(save_path_attn, map_location="cpu")

# 3. 双保险加载逻辑
try:
    # 尝试原位加载 (适用于 mg.model 已经经过剪枝 Pass 的情况)
    mg.model.load_state_dict(state_dict, strict=True)
    print("✅ 完美！权重完全匹配，原位加载成功！")
except RuntimeError as e:
    print("⚠️ 检测到 strict=True 报错，说明模型去掉了剪枝结构，尝试自动重映射 Key...")
    # 方案 A 的重映射逻辑：把 .parametrizations.weight.original 还原为 .weight
    fixed_state_dict = {}
    for k, v in state_dict.items():
        new_key = k.replace(".parametrizations.weight.original", ".weight")
        fixed_state_dict[new_key] = v
    mg.model.load_state_dict(fixed_state_dict, strict=False)
    print("✅ 已使用重映射模式加载成功！")

mg.model.to("cuda")
mg.model.eval()

# 4. 执行评估 (去掉了会报错的 per_device_eval_batch_size 参数)
print("🚀 开始对策略 C (注意力蒸馏) 进行终极评估...")
eval_trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy"
)

results = eval_trainer.evaluate()

# 5. 打印最终成绩单
print("\n" + "=" * 50)
print("📊 终极成绩单：注意力蒸馏 (策略 C)")
print("=" * 50)
print(f"🔹 历史基线 (仅 Logits): 0.83988")
print(f"🌟 当前模型 (Logits + Hidden + Attention): {results['eval_accuracy']:.5f}")
print("=" * 50)

🔄 正在读取权重: saved_models/kd_logits_hidden_attention/student_attention_kd_final2.0.pt
⚠️ 检测到 strict=True 报错，说明模型去掉了剪枝结构，尝试自动重映射 Key...
✅ 已使用重映射模式加载成功！
🚀 开始对策略 C (注意力蒸馏) 进行终极评估...


/workspace/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



📊 终极成绩单：注意力蒸馏 (策略 C)
🔹 历史基线 (仅 Logits): 0.83988
🌟 当前模型 (Logits + Hidden + Attention): 0.83556
